In [ ]:
import json
import csv
from pathlib import Path
from collections import defaultdict

In [ ]:
TARGET_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
# TARGET_MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
# TARGET_MODEL_ID = "zai-org/glm-4-9b-chat-hf"
# TARGET_MODEL_ID = "internlm/internlm3-8b-instruct"
# TARGET_MODEL_ID = "mistralai/Ministral-3-8B-Instruct-2512"

In [ ]:
SA_DATASET_NAMES = ["DoS", "Fuzzy", "Gear", "RPM"]

SA_QUESTION_FILES = {
    "DoS":   Path("DoS_sa_qa/questions/dos_sa_questions.json"),
    "Fuzzy": Path("Fuzzy_sa_qa/questions/fuzzy_sa_questions.json"),
    "Gear":  Path("Gear_sa_qa/questions/gear_sa_questions.json"),
    "RPM":   Path("RPM_sa_qa/questions/rpm_sa_questions.json"),
}

MODEL_TAG = TARGET_MODEL_ID.split("/")[-1].replace(".", "_").replace("-", "_")

def sa_answer_path(dataset_name: str, model_tag: str) -> Path:
    return Path(f"{dataset_name}_sa_qa/llm_answers/{dataset_name.lower()}_sa_answers_{model_tag}.jsonl")



In [ ]:
def load_sa_questions_map(path: Path):
    """
    Returns {qa_id: ground_truth_answer}
    Accepts .json (list) or .jsonl inputs.
    """
    if not path.exists():
        return {}
    if path.suffix == ".json":
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    data.append(json.loads(line))
    qmap = {}
    for rec in data:
        qa_id = rec.get("qa_id")
        answer = rec.get("ground_truth", rec.get("answer", ""))
        if qa_id is not None:
            qmap[qa_id] = answer
    return qmap


def load_sa_answers(path: Path):
    records = []
    if not path.exists():
        return records
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def normalize_text(text: str) -> str:
    return " ".join(text.strip().split()).lower() if text else ""



In [ ]:
def compute_sa_records(qmap, answers):
    """
    Build evaluation records with normalized predictions and gold answers.
    """
    eval_records = []
    for rec in answers:
        qa_id = rec.get("qa_id")
        if qa_id is None:
            continue
        pred_raw = rec.get("llm_answer", "")
        gold_raw = rec.get("ground_truth") or qmap.get(qa_id, "")
        pred = normalize_text(pred_raw)
        gold = normalize_text(gold_raw)
        eval_records.append({
            "qa_id": qa_id,
            "pred_raw": pred_raw,
            "gold_raw": gold_raw,
            "pred": pred,
            "gold": gold,
        })
    return eval_records


def accuracy_from_records(records):
    if not records:
        return 0.0, 0, 0
    total = len(records)
    correct = sum(int(r["pred"] == r["gold"]) for r in records)
    return correct / total if total else 0.0, correct, total



In [ ]:
OUTPUT_DIR = Path("LLM_SA_Performance")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_csv = OUTPUT_DIR / f"Performance_SA_{MODEL_TAG}.csv"

header = ["Attack", "Total", "Correct", "Accuracy"]
all_records = []

with out_csv.open("w", encoding="utf-8", newline="") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(header)

    for name in SA_DATASET_NAMES:
        q_path = SA_QUESTION_FILES[name]
        a_path = sa_answer_path(name, MODEL_TAG)

        qmap = load_sa_questions_map(q_path)
        ans_records = load_sa_answers(a_path)

        if not qmap:
            print(f"[WARN] Skip {name}: questions missing at {q_path}.")
            continue
        if not ans_records:
            print(f"[WARN] Skip {name}: answers missing at {a_path}.")
            continue

        eval_records = compute_sa_records(qmap, ans_records)
        acc, correct, total = accuracy_from_records(eval_records)
        writer.writerow([name, total, correct, f"{acc:.3f}"])

        all_records.extend(eval_records)

    if all_records:
        acc, correct, total = accuracy_from_records(all_records)
        writer.writerow(["Combined", total, correct, f"{acc:.3f}"])

print(f"[INFO] SA performance saved to {out_csv}")


[INFO] MCQ performance saved to LLM_MCQ_Performce/Performance_MCQ_DeepSeek_R1_Distill_Llama_8B.csv
